# exp-014: Wilcoxon 부호순위 검정 — setup 페어 통계 유의성

- **목적:** 기존 exp_results_NDCG10.csv (10 setups × 5 관점) + exp-006 25셀 매트릭스 위에서 setup 페어의 NDCG@10 차이가 통계적으로 유의한지 검정.
- **차별성 축:** ④ GT 부재 평가 신뢰도
- **데이터:** 
  - `raw/data/gemini_profile_outputs/exp_results_NDCG10.csv` (S01~S10 × A~E)
  - `raw/data/gemini_profile_outputs/exp_results_long.csv` (long-form)
- **메트릭:** Wilcoxon p-value (양측), Cohen's d (effect size), Bonferroni 보정
- **한계 명시:** N=5 paired observations (5 관점) — 검정력 낮음. exp-008 (N=30) 확장 후 재실행 필요.
- **관련 위키:** 100쌍-벤치마크-결과분석, dual-encoder-진화-실험계획 exp-014
- **작성일/실행일:** 2026-05-18
- **풀데이터 필요?** ❌ 불필요 — 기존 NDCG@10 결과만 사용

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import json
from itertools import combinations

DATA = Path('raw/data/gemini_profile_outputs')
OUT_DIR = Path('raw/experiments/exp-014-wilcoxon')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# NDCG@10 매트릭스 (10 setups × 5 관점) 로드
ndcg10 = pd.read_csv(DATA / 'exp_results_NDCG10.csv')
print('=== exp_results_NDCG10.csv ===')
print(ndcg10.round(4).to_string(index=False))

=== exp_results_NDCG10.csv ===
setup_id      A      B      C      D      E   mean
     S09 0.9975 0.9386 0.9643 0.9332 0.9940 0.9656
     S10 0.9924 0.9312 0.9445 0.9462 0.9913 0.9611
     S05 0.9686 0.9346 0.9484 0.9242 0.9645 0.9481
     S07 0.9503 0.9155 0.9266 0.9462 0.9406 0.9358
     S01 0.9584 0.9277 0.9020 0.9269 0.9298 0.9290
     S02 0.9327 0.9190 0.9001 0.9426 0.9276 0.9244
     S08 0.9433 0.9018 0.8983 0.9283 0.9394 0.9222
     S06 0.9391 0.8970 0.9102 0.9405 0.8889 0.9151
     S03 0.9270 0.8879 0.9190 0.9218 0.9119 0.9135
     S04 0.9330 0.8992 0.9042 0.8773 0.9257 0.9079


In [3]:
# Cohen's d (paired) 계산 함수
def cohens_d_paired(x, y):
    diff = np.array(x) - np.array(y)
    return diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else 0.0

def effect_size_label(d):
    d = abs(d)
    if d < 0.2: return 'negligible'
    elif d < 0.5: return 'small'
    elif d < 0.8: return 'medium'
    else: return 'large'

# setup 페어별 Wilcoxon (5 관점 paired)
setups = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']
perspectives = ['A', 'B', 'C', 'D', 'E']

ndcg10_pivot = ndcg10.set_index('setup_id')[perspectives]

results = []
for s_i, s_j in combinations(setups, 2):
    x = ndcg10_pivot.loc[s_i, perspectives].values
    y = ndcg10_pivot.loc[s_j, perspectives].values
    diff = x - y
    # zsplit handles zero differences
    try:
        stat, p = stats.wilcoxon(x, y, alternative='two-sided', zero_method='wilcox', method='exact')
    except ValueError:
        stat, p = np.nan, 1.0
    d = cohens_d_paired(x, y)
    results.append({
        'pair': f'{s_i} vs {s_j}',
        'setup_i': s_i, 'setup_j': s_j,
        'mean_i': float(x.mean()), 'mean_j': float(y.mean()),
        'mean_diff': float(diff.mean()),
        'wilcoxon_stat': float(stat) if not np.isnan(stat) else None,
        'p_value': float(p),
        'cohens_d': float(d),
        'effect_size': effect_size_label(d),
        'N_paired': 5,
    })

wilc_df = pd.DataFrame(results)

# Bonferroni 보정 (45 pair tests)
n_tests = len(wilc_df)
wilc_df['p_bonferroni'] = (wilc_df['p_value'] * n_tests).clip(upper=1.0)
wilc_df['significant_p05'] = wilc_df['p_value'] < 0.05
wilc_df['significant_bonferroni'] = wilc_df['p_bonferroni'] < 0.05

print(f'\n=== Wilcoxon (S01~S10 pairwise, N=5 관점 paired, {n_tests} pairs) ===')
print(wilc_df.sort_values('p_value').head(15).round(4).to_string(index=False))


=== Wilcoxon (S01~S10 pairwise, N=5 관점 paired, 45 pairs) ===
      pair setup_i setup_j  mean_i  mean_j  mean_diff  wilcoxon_stat  p_value  cohens_d effect_size  N_paired  p_bonferroni  significant_p05  significant_bonferroni
S03 vs S09     S03     S09  0.9135  0.9656    -0.0520            0.0   0.0625   -1.9176       large         5           1.0            False                   False
S06 vs S07     S06     S07  0.9151  0.9358    -0.0207            0.0   0.0625   -1.1473       large         5           1.0            False                   False
S06 vs S10     S06     S10  0.9151  0.9611    -0.0460            0.0   0.0625   -1.2829       large         5           1.0            False                   False
S03 vs S07     S03     S07  0.9135  0.9358    -0.0223            0.0   0.0625   -2.6250       large         5           1.0            False                   False
S07 vs S08     S07     S08  0.9358  0.9222     0.0136            0.0   0.0625    1.3072       large         5    

In [4]:
# S04 (KR-SBERT baseline) vs 나머지 — 가장 중요한 비교
print('=== S04 (Baseline KR-SBERT raw) vs 나머지 9 setups ===')
s04_pairs = wilc_df[(wilc_df['setup_i']=='S04') | (wilc_df['setup_j']=='S04')].copy()
# normalize: S04를 항상 i로
for idx, row in s04_pairs.iterrows():
    if row['setup_j'] == 'S04':
        # swap
        s04_pairs.at[idx, 'setup_i'] = row['setup_j']
        s04_pairs.at[idx, 'setup_j'] = row['setup_i']
        s04_pairs.at[idx, 'mean_i'] = row['mean_j']
        s04_pairs.at[idx, 'mean_j'] = row['mean_i']
        s04_pairs.at[idx, 'mean_diff'] = -row['mean_diff']
        s04_pairs.at[idx, 'cohens_d'] = -row['cohens_d']
s04_pairs['Δ_vs_S04'] = s04_pairs['mean_j'] - s04_pairs['mean_i']
print(s04_pairs[['setup_j', 'mean_j', 'Δ_vs_S04', 'wilcoxon_stat', 'p_value', 'cohens_d', 'effect_size', 'significant_p05']].round(4).to_string(index=False))

=== S04 (Baseline KR-SBERT raw) vs 나머지 9 setups ===
setup_j  mean_j  Δ_vs_S04  wilcoxon_stat  p_value  cohens_d effect_size  significant_p05
    S01  0.9290    0.0211            1.0   0.1250   -1.0177       large            False
    S02  0.9244    0.0165            4.0   0.4375   -0.5733      medium            False
    S03  0.9135    0.0057            6.0   0.8125   -0.2309       small            False
    S05  0.9481    0.0402            0.0   0.0625   -7.7512       large            False
    S06  0.9151    0.0073            5.0   0.6250   -0.2022       small            False
    S07  0.9358    0.0280            0.0   0.0625   -1.2129       large            False
    S08  0.9222    0.0144            2.0   0.1875   -0.6566      medium            False
    S09  0.9656    0.0577            0.0   0.0625   -5.1528       large            False
    S10  0.9611    0.0532            0.0   0.0625   -3.2762       large            False


In [5]:
# S09 (best — profile+e5+overlap fusion) vs S07 (D 관점 1위)
s09_s07 = wilc_df[((wilc_df['setup_i']=='S09') & (wilc_df['setup_j']=='S07')) | 
                   ((wilc_df['setup_i']=='S07') & (wilc_df['setup_j']=='S09'))]
print('=== S09 (best 4-of-5) vs S07 (D 관점 1위) ===')
print(s09_s07.round(4).to_string(index=False))
print('\n관점별 NDCG@10:')
comp = ndcg10_pivot.loc[['S09', 'S07']]
comp['diff'] = comp.loc['S09'] - comp.loc['S07']
print(comp.round(4))

=== S09 (best 4-of-5) vs S07 (D 관점 1위) ===
      pair setup_i setup_j  mean_i  mean_j  mean_diff  wilcoxon_stat  p_value  cohens_d effect_size  N_paired  p_bonferroni  significant_p05  significant_bonferroni
S07 vs S09     S07     S09  0.9358  0.9656    -0.0297            1.0    0.125   -1.1238       large         5           1.0            False                   False

관점별 NDCG@10:
               A       B       C       D       E  diff
setup_id                                              
S09       0.9975  0.9386  0.9643  0.9332  0.9940   NaN
S07       0.9503  0.9155  0.9266  0.9462  0.9406   NaN


In [6]:
# 결과 저장
wilc_df.to_csv(OUT_DIR / 'wilcoxon_all_pairs.csv', index=False)
s04_pairs.to_csv(OUT_DIR / 'wilcoxon_vs_baseline_S04.csv', index=False)

# 요약
summary = {
    'exp_id': 'exp-014',
    'title': 'Wilcoxon 부호순위 검정 (10 setups × 5 관점)',
    'date': '2026-05-18',
    'n_pairs_tested': int(len(wilc_df)),
    'n_paired_observations': 5,
    'n_significant_p05': int(wilc_df['significant_p05'].sum()),
    'n_significant_bonferroni': int(wilc_df['significant_bonferroni'].sum()),
    'best_setup': str(ndcg10.set_index('setup_id')['mean'].idxmax()),
    'best_setup_mean_ndcg10': float(ndcg10.set_index('setup_id')['mean'].max()),
    'worst_setup': str(ndcg10.set_index('setup_id')['mean'].idxmin()),
    'worst_setup_mean_ndcg10': float(ndcg10.set_index('setup_id')['mean'].min()),
    'S09_vs_S04_p': float(wilc_df[(wilc_df['pair']=='S04 vs S09') | (wilc_df['pair']=='S09 vs S04')]['p_value'].iloc[0]),
    'S09_vs_S04_cohens_d': float(s04_pairs[s04_pairs['setup_j']=='S09']['cohens_d'].iloc[0]),
    'limitation': 'N=5 paired observations (5 관점) — 검정력 낮음. Wilcoxon exact min p ≈ 0.0625 for N=5 → 거의 유의 불가능. exp-008(N=30) 필요',
}
with open(OUT_DIR / 'exp014_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "exp_id": "exp-014",
  "title": "Wilcoxon 부호순위 검정 (10 setups × 5 관점)",
  "date": "2026-05-18",
  "n_pairs_tested": 45,
  "n_paired_observations": 5,
  "n_significant_p05": 0,
  "n_significant_bonferroni": 0,
  "best_setup": "S09",
  "best_setup_mean_ndcg10": 0.9655529284922622,
  "worst_setup": "S04",
  "worst_setup_mean_ndcg10": 0.9078754367366833,
  "S09_vs_S04_p": 0.0625,
  "S09_vs_S04_cohens_d": -5.152841569357057,
  "limitation": "N=5 paired observations (5 관점) — 검정력 낮음. Wilcoxon exact min p ≈ 0.0625 for N=5 → 거의 유의 불가능. exp-008(N=30) 필요"
}


In [7]:
# === 보조: exp-006 25셀 per-user-NDCG로 Wilcoxon (N=10, 더 검정력 ↑) ===
# 이건 weight set 페어 검정 — exp-006의 fusion 가중치 5세트 + SINGLE의 차이가 유의한가?
# 데이터: benchmark_labeled_100_{A,B,C,D,E} + weighted_results.csv
labels = {}
for p in 'ABCDE':
    df = pd.read_csv(DATA / f'benchmark_labeled_100_{p}.csv', encoding='utf-8-sig')
    df.columns = [c.lstrip('\ufeff') for c in df.columns]
    df['job_id'] = df['job_id'].astype(str)
    labels[p] = df

wr = pd.read_csv(DATA / 'weighted_results.csv', encoding='utf-8-sig')
wr.columns = [c.lstrip('\ufeff') for c in wr.columns]
wr['job_id'] = wr['job_id'].astype(str)
fusion_cols = ['role_semantic', 'hard_skill', 'competency', 'achievement', 'industry', 'quality_adjustment']

WEIGHTS = {
    'A': {'role_semantic': 0.50, 'hard_skill': 0.15, 'competency': 0.10, 'achievement': 0.05, 'industry': 0.10, 'quality_adjustment': 0.10},
    'B': {'role_semantic': 0.20, 'hard_skill': 0.15, 'competency': 0.10, 'achievement': 0.35, 'industry': 0.10, 'quality_adjustment': 0.10},
    'C': {'role_semantic': 0.15, 'hard_skill': 0.45, 'competency': 0.15, 'achievement': 0.05, 'industry': 0.10, 'quality_adjustment': 0.10},
    'D': {'role_semantic': 0.15, 'hard_skill': 0.10, 'competency': 0.05, 'achievement': 0.05, 'industry': 0.55, 'quality_adjustment': 0.10},
    'E': {'role_semantic': 0.20, 'hard_skill': 0.20, 'competency': 0.15, 'achievement': 0.15, 'industry': 0.20, 'quality_adjustment': 0.10},
    'SINGLE': {'role_semantic': 0.35, 'hard_skill': 0.20, 'competency': 0.15, 'achievement': 0.10, 'industry': 0.10, 'quality_adjustment': 0.10},
}

def ndcg_at_k(rels, k=10):
    rels = np.asarray(rels, dtype=float)
    if len(rels) == 0: return 0.0
    rels_k = rels[:k]
    gains = (2**rels_k - 1) / np.log2(np.arange(2, len(rels_k)+2))
    ideal = np.sort(rels)[::-1][:k]
    ideal_gains = (2**ideal - 1) / np.log2(np.arange(2, len(ideal)+2))
    return gains.sum() / ideal_gains.sum() if ideal_gains.sum() > 0 else 0.0

# per-user NDCG@10 계산
per_user_ndcg = {w: {} for w in WEIGHTS}
for w_code, weights in WEIGHTS.items():
    for eval_code in 'ABCDE':
        L = labels[eval_code]
        M = L.merge(wr[['userId', 'job_id'] + fusion_cols], on=['userId', 'job_id'], how='left', suffixes=('_label', ''))
        for c in fusion_cols:
            if c in M.columns: M[c] = M[c].fillna(0)
            else: M[c] = 0
        M['score'] = sum(M[c] * weights[c] for c in fusion_cols)
        user_ndcg = []
        for uid, g in M.groupby('userId'):
            g_sorted = g.sort_values('score', ascending=False)
            user_ndcg.append(ndcg_at_k(g_sorted['judge_relevance'].fillna(0).values, 10))
        per_user_ndcg[w_code][eval_code] = user_ndcg

# weight set 페어 Wilcoxon (관점별 + 통합)
weight_pair_results = []
for w_i, w_j in combinations(['A', 'B', 'C', 'D', 'E', 'SINGLE'], 2):
    # 통합: 모든 관점 user NDCG concat (관점이 다르면 user도 다르지만 paired 가능 - 같은 user에 weight만 다름)
    x_all, y_all = [], []
    for ec in 'ABCDE':
        xi = per_user_ndcg[w_i][ec]
        yi = per_user_ndcg[w_j][ec]
        n = min(len(xi), len(yi))
        x_all.extend(xi[:n])
        y_all.extend(yi[:n])
    if len(x_all) > 0:
        try:
            stat, p = stats.wilcoxon(x_all, y_all, alternative='two-sided', zero_method='wilcox')
        except ValueError:
            stat, p = np.nan, 1.0
        d = cohens_d_paired(x_all, y_all)
        weight_pair_results.append({
            'pair': f'{w_i} vs {w_j}',
            'N_paired': len(x_all),
            'mean_i': float(np.mean(x_all)),
            'mean_j': float(np.mean(y_all)),
            'mean_diff': float(np.mean(x_all) - np.mean(y_all)),
            'wilcoxon_p': float(p),
            'cohens_d': float(d),
            'effect_size': effect_size_label(d),
            'significant_p05': p < 0.05,
        })

wp_df = pd.DataFrame(weight_pair_results)
wp_df.to_csv(OUT_DIR / 'wilcoxon_weight_pairs.csv', index=False)
print('\n=== weight 페어 Wilcoxon (50 user-perspective trials) ===')
print(wp_df.round(4).to_string(index=False))


=== weight 페어 Wilcoxon (50 user-perspective trials) ===
       pair  N_paired  mean_i  mean_j  mean_diff  wilcoxon_p  cohens_d effect_size  significant_p05
     A vs B        50  0.9523  0.9527    -0.0004      0.8658   -0.0149  negligible            False
     A vs C        50  0.9523  0.9512     0.0011      0.8886    0.0357  negligible            False
     A vs D        50  0.9523  0.9562    -0.0039      0.2850   -0.1794  negligible            False
     A vs E        50  0.9523  0.9536    -0.0013      0.6547   -0.1092  negligible            False
A vs SINGLE        50  0.9523  0.9513     0.0010      1.0000    0.0438  negligible            False
     B vs C        50  0.9527  0.9512     0.0015      0.6547    0.1165  negligible            False
     B vs D        50  0.9527  0.9562    -0.0035      0.4631   -0.1136  negligible            False
     B vs E        50  0.9527  0.9536    -0.0009      0.8927   -0.0364  negligible            False
B vs SINGLE        50  0.9527  0.9513     0